# 7B vs 125M on CORD-v2 - GPU steps

Runtime: **A100**. About 50 minutes for pass A, 20 for pass B.

**This notebook is two passes, in two separate runtimes, and that is not optional.**
Installing vLLM replaces torch with a CUDA 13 build while Colab's preinstalled
torchaudio is CUDA 12.8. `transformers` imports torchaudio, so after a vLLM install
nothing else in this project can import at all. Pass A produces every accuracy number
and the transformers benchmark; pass B produces the vLLM benchmark in a runtime it is
allowed to own. Download pass A's artifacts before starting pass B.

The harness checks run first, before any GPU time is spent. If the converter ceiling
is not exactly 1.000, no model number produced later is worth recording.

---
# Pass A - accuracy and the transformers benchmark

In [ ]:
import os
!git clone -q https://github.com/Perlious-Savage/qwen-qlora-vs-layoutlmv3.git
os.chdir('/content/qwen-qlora-vs-layoutlmv3')
!pip install -q -r requirements-train.txt

# PEFT's torchao dispatcher raises rather than returning False when torchao is older
# than it wants, which breaks loading any adapter onto a bf16 model. Nothing here uses
# torchao, so remove it rather than upgrading it - upgrading drags in a new torch.
!pip uninstall -y -q torchao

## 0. Verify the harness before spending GPU time

In order of how badly each one invalidates the result: the scoring code is
byte-identical to Project 1's; the JSON-to-spans converter loses nothing; no test
document also appears in train.

In [ ]:
!sha256sum -c VENDORED.sha256
!pytest tests/ -q

In [ ]:
!python roundtrip_check.py

In [ ]:
!python scripts/check_splits.py
!python -m src.eval_llm --lengths

## 1. Base model, 2-shot

The number the fine-tune has to beat. Read `parse_rate` before `test_f1`: a low F1
with a low parse rate means the model never followed the output contract, which is a
different finding from reading receipts badly.

In [ ]:
!python -m src.eval_llm --config base --fewshot 2 --batch-size 4

## 2. QLoRA fine-tune

Smoke test first, into a throwaway output directory so it cannot clobber the real
adapter. **Read its output before running the next cell** - a failure here does not
stop the real run, because `!` lines in a notebook do not short-circuit.

In [ ]:
!python -m src.train_qlora --max-train 40 --epochs 1 --output outputs/smoke

In [ ]:
!python -m src.train_qlora --epochs 3

Get the adapter off the runtime immediately. It is 92 MB, it is gitignored on
purpose, and a disconnect here costs 15 minutes of training. Pass B needs it too.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r outputs/qwen-cord-lora /content/drive/MyDrive/

## 3. Score the fine-tune - same split, same decoding config

In [ ]:
!python -m src.eval_llm --config qlora --adapter outputs/qwen-cord-lora

## 4. Benchmark

All configurations in one process on one GPU so the rows are comparable. The
LayoutLMv3 row is a different workload - one forward pass, not generation - and the
table says so.

In [ ]:
!python bench.py --adapter outputs/qwen-cord-lora --documents 10

## 5. Take the results with you

Do this before pass B. Pass B's install destroys this runtime's environment, and
anything not downloaded here is lost.

In [ ]:
!python compare.py
!zip -qr /content/pass_a.zip artifacts/ mlruns/
from google.colab import files
files.download('/content/pass_a.zip')

---
# Pass B - vLLM serving benchmark

**Runtime -> Disconnect and delete runtime first.** Then run the cells below in the
fresh runtime.

vLLM is installed *first* here, and the mismatched CUDA 12.8 companions are removed
*before* it lands, so it can bring its own torch without leaving a broken import
behind. Nothing from pass A's requirements is installed - vLLM supplies transformers
itself, and only `datasets` is added on top.

In [ ]:
import os
!git clone -q https://github.com/Perlious-Savage/qwen-qlora-vs-layoutlmv3.git
os.chdir('/content/qwen-qlora-vs-layoutlmv3')
!pip uninstall -y -q torchaudio torchvision torchao
!pip install -q vllm
!pip install -q datasets bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p outputs && cp -r /content/drive/MyDrive/qwen-cord-lora outputs/

Each configuration runs in isolation, so one failing does not discard the ones that
already measured. On the run recorded in this repo the 4-bit row failed - that vLLM
build rejects `bitsandbytes` quantization - and its error is recorded in place of its
numbers rather than the row silently disappearing.

In [ ]:
!python bench.py --adapter outputs/qwen-cord-lora --engine vllm --skip-encoder --documents 10

In [ ]:
!python compare.py
!zip -qr /content/pass_b.zip artifacts/
from google.colab import files
files.download('/content/pass_b.zip')

---
## Optional: LoRA rank sweep

Adds roughly three hours to pass A. Not run for the results in this repo, and the
README says so rather than implying a sensitivity table exists.

In [ ]:
# !python sweep.py --fast